# 01 — EDA: Industrial Failure Classification

Dataset: AI4I 2020 Predictive Maintenance (UCI)
https://archive.ics.uci.edu/dataset/601/ai4i+2020+predictive+maintenance+dataset

Goals:
- Understand class imbalance (~3.4% failure rate)
- Visualize feature distributions by failure class
- Identify which sensors correlate most with failure
- Understand failure modes (TWF, HDF, PWF, OSF, RNF)

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
import sys; sys.path.insert(0, '..')
from src.features import download_data, load_raw, build_features, TARGET_COL, FAILURE_MODE_COLS

%matplotlib inline
sns.set_theme(style='darkgrid', palette='muted')

In [ ]:
# Download data if needed
download_data()
df = load_raw()
print(df.shape)
df.head()

In [ ]:
# Class distribution
fail_counts = df[TARGET_COL].value_counts()
print(f"No Failure: {fail_counts[0]:,} ({fail_counts[0]/len(df)*100:.1f}%)")
print(f"Failure:    {fail_counts[1]:,} ({fail_counts[1]/len(df)*100:.1f}%)")

fig, ax = plt.subplots(figsize=(5, 3))
fail_counts.plot(kind='bar', ax=ax, color=['steelblue', 'tomato'])
ax.set_xticklabels(['No Failure', 'Failure'], rotation=0)
ax.set_title('Class Distribution (highly imbalanced)')
plt.tight_layout(); plt.show()

In [ ]:
# Failure mode breakdown
mode_counts = df[FAILURE_MODE_COLS].sum().sort_values(ascending=False)
print("Failure modes:")
print(mode_counts)

In [ ]:
# Feature distributions: failure vs. no failure
raw_features = ['Air temperature [K]', 'Process temperature [K]',
                'Rotational speed [rpm]', 'Torque [Nm]', 'Tool wear [min]']

fig, axes = plt.subplots(2, 3, figsize=(14, 7))
axes = axes.flatten()

for i, feat in enumerate(raw_features):
    for label, color in [(0, 'steelblue'), (1, 'tomato')]:
        axes[i].hist(df[df[TARGET_COL]==label][feat], bins=40, alpha=0.6,
                     label=f"{'Failure' if label else 'No Failure'}", color=color, density=True)
    axes[i].set_title(feat)
    axes[i].legend(fontsize=8)

axes[-1].axis('off')
plt.suptitle('Feature Distributions: Failure vs. No Failure', fontsize=12)
plt.tight_layout(); plt.show()

In [ ]:
# Point biserial correlations with target
from scipy.stats import pointbiserialr

corr_data = []
for feat in raw_features:
    r, p = pointbiserialr(df[TARGET_COL], df[feat])
    corr_data.append({'feature': feat, 'correlation': r, 'p_value': p})

corr_df = pd.DataFrame(corr_data).sort_values('correlation', ascending=False)
print(corr_df.to_string(index=False))
# Expected: Tool wear and Torque will be highest

In [ ]:
# Engineered features
df_feat = build_features(df)
print(df_feat[['temp_diff', 'power', 'wear_rate', 'torque_per_wear', 'high_load']].describe())

In [ ]:
# Failure rate by type
type_fail = df.groupby('Type')[TARGET_COL].mean() * 100
print("Failure rate by machine type:")
print(type_fail.round(2))